In [ ]:
# Parameters
WORKSPACE_ID   = "990dbc7b-f5d1-4bc8-a929-9dfd509a5d52"
LAKEHOUSE_ID   = "ec851642-fa89-42bc-aebf-2742845d36fe"
WAREHOUSE_CONN = ""  # optional — leave blank to skip warehouse write
SCALE_SIZE     = "small"
SEED           = 42


In [ ]:
from sqllocks_spindle import Spindle
from sqllocks_spindle.domains.retail import RetailDomain
from sqllocks_spindle.domains.financial import FinancialDomain
from azure.identity import InteractiveBrowserCredential
from deltalake import write_deltalake

SOUND_BI_TENANT = "2536810f-20e1-4911-a453-4409fd96db8a"
token = InteractiveBrowserCredential(tenant_id=SOUND_BI_TENANT).get_token("https://storage.azure.com/.default").token
storage_opts = {"bearer_token": token, "use_emulator": "false"}

spindle = Spindle()
retail_result    = spindle.generate(domain=RetailDomain(),    scale=SCALE_SIZE, seed=SEED)
financial_result = spindle.generate(domain=FinancialDomain(), scale=SCALE_SIZE, seed=SEED + 1)

print(f"Retail: {sum(len(df) for df in retail_result.tables.values()):,} rows")
print(f"Financial: {sum(len(df) for df in financial_result.tables.values()):,} rows")


In [ ]:
# Write both domains to lakehouse
for domain_prefix, result in [("retail", retail_result), ("financial", financial_result)]:
    for table_name, df in result.tables.items():
        path = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_ID}/Tables/{domain_prefix}_{table_name}"
        write_deltalake(path, df, mode="overwrite", storage_options=storage_opts, schema_mode="overwrite")
        print(f"  {domain_prefix}_{table_name}: {len(df):,} rows -> lakehouse")


In [ ]:
# Optionally write to warehouse too
if WAREHOUSE_CONN:
    from sqllocks_spindle.fabric.sql_database_writer import FabricSqlDatabaseWriter
    writer = FabricSqlDatabaseWriter(WAREHOUSE_CONN, auth_method="cli")
    writer.write(retail_result, schema_name="retail", mode="create_insert")
    writer.write(financial_result, schema_name="financial", mode="create_insert")
    print("Also written to Warehouse (retail + financial schemas)")
else:
    print("WAREHOUSE_CONN not set — skipping warehouse write")
